# Senologie Diagnose Analysis

Query Aidbox FHIR Server for breast cancer diagnoses.

In [ ]:
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth

AIDBOX_URL = "http://localhost:8888"
AUTH = HTTPBasicAuth("root", "secret")

In [ ]:
# Fetch all Conditions
response = requests.get(
    f"{AIDBOX_URL}/fhir/Condition",
    auth=AUTH,
    headers={"Accept": "application/fhir+json"}
)
bundle = response.json()
print(f"Found {bundle.get('total', len(bundle.get('entry', [])))} conditions")

In [ ]:
# Extract to DataFrame
records = []
for entry in bundle.get('entry', []):
    r = entry['resource']
    
    # Get SNOMED code
    snomed = next((c for c in r.get('code', {}).get('coding', []) 
                   if c.get('system') == 'http://snomed.info/sct'), {})
    
    # Get ICD-10
    icd10 = next((c for c in r.get('code', {}).get('coding', []) 
                  if 'icd-10' in c.get('system', '')), {})
    
    # Get laterality
    body_site = r.get('bodySite', [{}])[0].get('coding', [{}])[0]
    
    records.append({
        'id': r.get('id'),
        'patient': r.get('subject', {}).get('reference'),
        'snomed_code': snomed.get('code'),
        'snomed_display': snomed.get('display'),
        'icd10_code': icd10.get('code'),
        'laterality': body_site.get('display'),
        'clinical_status': r.get('clinicalStatus', {}).get('coding', [{}])[0].get('code'),
        'recorded_date': r.get('recordedDate')
    })

df = pd.DataFrame(records)
df

In [ ]:
# Diagnosis distribution
df['snomed_display'].value_counts()

In [ ]:
# Clinical status distribution
df['clinical_status'].value_counts()

## Aidbox SQL Query

Aidbox supports native SQL queries on FHIR resources.

In [ ]:
# SQL query via Aidbox $sql endpoint
sql = """
SELECT 
    id,
    resource#>>'{subject,reference}' as patient,
    resource#>>'{code,coding,0,code}' as snomed_code,
    resource#>>'{code,coding,0,display}' as diagnosis,
    resource#>>'{bodySite,0,coding,0,display}' as laterality,
    resource#>>'{clinicalStatus,coding,0,code}' as status,
    resource->>'recordedDate' as recorded_date
FROM condition
"""

response = requests.post(
    f"{AIDBOX_URL}/$sql",
    auth=AUTH,
    json=[sql]
)
sql_df = pd.DataFrame(response.json())
sql_df